# Passo 2 - Pré-processamento de Dados e Criação da Variável de Engajamento

Este notebook tem como objetivo preparar os dados para o desenvolvimento do modelo de previsão de engajamento dos usuários no projeto WellMe.

Nesta etapa, serão realizadas ações como seleção de variáveis relevantes, tratamento de dados, criação da variável alvo e organização do dataset final que será utilizado no treinamento do modelo MLP.

Como o WellMe ainda não possui dados reais disponíveis, a variável de engajamento será construída com base em regras definidas a partir dos comportamentos presentes no dataset, considerando indicadores como atividade física, tempo sedentário e gasto calórico.

In [25]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv('../data/raw/fitbit_daily_activity.csv')

df.head()

,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,1503960366,3/25/2016,11004,7.11,7.11,0.0,2.57,0.46,4.07,0.0,33,12,205,804,1819
1,1503960366,3/26/2016,17609,11.55,11.55,0.0,6.92,0.73,3.91,0.0,89,17,274,588,2154
2,1503960366,3/27/2016,12736,8.53,8.53,0.0,4.66,0.16,3.71,0.0,56,5,268,605,1944
3,1503960366,3/28/2016,13231,8.93,8.93,0.0,3.19,0.79,4.95,0.0,39,20,224,1080,1932
4,1503960366,3/29/2016,12041,7.85,7.85,0.0,2.16,1.09,4.61,0.0,28,28,243,763,1886


In [27]:
selected_columns = [
    'Id',
    'ActivityDate',
    'TotalSteps',
    'VeryActiveMinutes',
    'FairlyActiveMinutes',
    'LightlyActiveMinutes',
    'SedentaryMinutes',
    'Calories'
]

df_selected = df[selected_columns].copy()

df_selected.head()

,Id,ActivityDate,TotalSteps,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,1503960366,3/25/2016,11004,33,12,205,804,1819
1,1503960366,3/26/2016,17609,89,17,274,588,2154
2,1503960366,3/27/2016,12736,56,5,268,605,1944
3,1503960366,3/28/2016,13231,39,20,224,1080,1932
4,1503960366,3/29/2016,12041,28,28,243,763,1886


### Seleção de Variáveis

Nesta etapa, foram selecionadas as variáveis mais relevantes para representar o comportamento dos usuários no contexto do projeto WellMe.

Foram mantidas variáveis relacionadas à atividade física, como `TotalSteps`, `VeryActiveMinutes`, `FairlyActiveMinutes` e `LightlyActiveMinutes`, pois elas representam tanto a quantidade de movimento quanto diferentes intensidades de atividade.

A variável `SedentaryMinutes` também foi selecionada por representar o tempo de inatividade do usuário, sendo importante para diferenciar comportamentos mais ativos de comportamentos mais sedentários.

Além disso, a variável `Calories` foi mantida por indicar o gasto energético diário, complementando a análise do nível de atividade.

As colunas `Id` e `ActivityDate` foram preservadas para identificação dos usuários e registro temporal das atividades.

In [28]:
df_selected[['TotalSteps', 'SedentaryMinutes', 'Calories']].describe()

,TotalSteps,SedentaryMinutes,Calories
count,457.000000,457.000000,457.000000
mean,6546.562363,995.282276,2189.452954
std,5398.493064,337.021404,815.484523
min,0.000000,32.000000,0.000000
25%,1988.000000,728.000000,1776.000000
50%,5986.000000,1057.000000,2062.000000
75%,10198.000000,1285.000000,2667.000000
max,28497.000000,1440.000000,4562.000000


In [29]:
df_selected['steps_score'] = pd.qcut(df_selected['TotalSteps'], 3, labels=[0, 1, 2])

df_selected['sedentary_score'] = pd.qcut(
    df_selected['SedentaryMinutes'], 3, labels=[2, 1, 0]
)

df_selected['calories_score'] = pd.qcut(df_selected['Calories'], 3, labels=[0, 1, 2])

In [30]:
df_selected['engagement_score'] = (
    df_selected['steps_score'].astype(int) +
    df_selected['sedentary_score'].astype(int) +
    df_selected['calories_score'].astype(int)
)

In [32]:
def classify_engagement(score):
    if score >= 4:
        return 'High'
    elif score >= 2:
        return 'Medium'
    else:
        return 'Low'

df_selected['Engagement'] = df_selected['engagement_score'].apply(classify_engagement)

In [33]:
df_selected[['TotalSteps', 'SedentaryMinutes', 'Calories', 'Engagement']].head()

,TotalSteps,SedentaryMinutes,Calories,Engagement
0,11004,804,1819,Medium
1,17609,588,2154,High
2,12736,605,1944,High
3,13231,1080,1932,High
4,12041,763,1886,High


### Criação da Variável de Engajamento

Com base na análise estatística das variáveis, foi possível identificar a distribuição dos dados utilizando quartis (25%, 50% e 75%), permitindo uma divisão equilibrada dos usuários em diferentes níveis de comportamento.

A variável de engajamento foi construída a partir de três indicadores principais: `TotalSteps`, `SedentaryMinutes` e `Calories`.

Cada variável foi dividida em três grupos (baixo, médio e alto) utilizando percentis, garantindo que a classificação fosse proporcional ao próprio dataset.

A variável `SedentaryMinutes` foi tratada de forma inversa, considerando que maiores valores representam menor engajamento.

A partir dessas classificações, foi criado um score de engajamento, que posteriormente foi convertido em três níveis: baixo, médio e alto.

In [34]:
df_selected['Engagement'].value_counts()

Engagement
High      199
Medium    143
Low       115
Name: count, dtype: int64

### Distribuição da Variável de Engajamento

A distribuição da variável `Engagement` mostra a quantidade de registros classificados em cada nível de engajamento.

Observa-se que a classe "High" possui maior número de registros, seguida por "Medium" e "Low". Apesar dessa diferença, as três classes apresentam quantidades razoavelmente próximas, não havendo um desbalanceamento significativo.

Essa distribuição é adequada para o treinamento de modelos de Machine Learning, pois evita que o modelo fique enviesado para uma única classe.

Dessa forma, a variável de engajamento construída apresenta uma distribuição satisfatória, permitindo seu uso como variável alvo no desenvolvimento do modelo de previsão.

In [35]:
df_final = df_selected.drop(columns=[
    'steps_score',
    'sedentary_score',
    'calories_score',
    'engagement_score'
])

df_final.head()

,Id,ActivityDate,TotalSteps,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Engagement
0,1503960366,3/25/2016,11004,33,12,205,804,1819,Medium
1,1503960366,3/26/2016,17609,89,17,274,588,2154,High
2,1503960366,3/27/2016,12736,56,5,268,605,1944,High
3,1503960366,3/28/2016,13231,39,20,224,1080,1932,High
4,1503960366,3/29/2016,12041,28,28,243,763,1886,High


In [36]:
df_final.to_csv('../data/processed/fitbit_processed.csv', index=False)

### Limpeza e Salvamento dos Dados

Após a criação da variável de engajamento, foram removidas as colunas auxiliares utilizadas apenas para cálculo, mantendo no dataset apenas as variáveis relevantes para o modelo.

Em seguida, o dataset final foi salvo na pasta `data/processed`, separando os dados tratados dos dados brutos.

Essa etapa é importante para manter a organização do projeto e facilitar o uso dos dados nas próximas fases, especialmente no treinamento do modelo.